# 07 - Behavior Classification
Train an XGBoost classifier on pose features to classify behaviors.

In [ ]:
# ===== CONFIGURATION =====
GITHUB_REPO_URL = "https://github.com/kaarthik-balakrishnan/LightningPoseTrack.git"
GIT_BRANCH = "main"

DRIVE_ROOT = "/content/drive/My Drive/PigBehavior"
DRIVE_FEATURES = f"{DRIVE_ROOT}/features"
DRIVE_MODELS = f"{DRIVE_ROOT}/trained_models"

# Feature window sizes (seconds)
WINDOW_SIZES = [1, 3, 5]  # seconds
FPS = 30.0

# Model params
TEST_SIZE = 0.2
RANDOM_STATE = 42
DRIVE_FOLDER_ID = "1X_41ZW3HfwVeft2lPld3XNqXsdxRDIwb"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, sys
REPO_DIR = "/content/LightningPoseTrack"
if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

In [ ]:
!pip install --quiet pandas numpy scikit-learn xgboost pyarrow imageio[ffmpeg]

In [ ]:
print("=" * 60)
print("LABELING INSTRUCTION:")
print("=" * 60)
print("Before training, you need labeled behavior data.")
print("\nOption 1: Manual labeling via a labeling tool")
print("  - Use the interactive labeling helper below")
print("  - Load pose trajectories and assign behavior labels frame-by-frame")
print("\nOption 2: Use feeding events as labels (auto)")
print("  - Feeding events from notebook 06 can be used as positive labels")
print("  - Other frames can be sampled as negative examples")
print("\nLabels: feeding, standing, walking, turning, resting")

In [ ]:
# Interactive labeling helper
# This generates a CSV template for labeling
from pathlib import Path
import pandas as pd

features_dir = Path(DRIVE_FEATURES)
feature_files = list(features_dir.rglob("*.parquet"))

if feature_files:
    sample_df = pd.read_parquet(feature_files[0])
    n_frames = len(sample_df)
    # Create a labeling template
    label_template = pd.DataFrame({
        "frame": range(n_frames),
        "timestamp": [f"{i/FPS:.2f}" for i in range(n_frames)],
        "behavior": [""] * n_frames,
    })
    template_path = Path(DRIVE_ROOT) / "labeling_template.csv"
    label_template.to_csv(template_path, index=False)
    print(f"Labeling template saved to {template_path}")
    print("\nFill in the 'behavior' column with: feeding, standing, walking, turning, resting")
    print("Then run the training cell below.")

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import xgboost as xgb
import joblib

# Load labeled data
labels_path = Path(DRIVE_ROOT) / "labeling_template.csv"
if not labels_path.exists():
    print("No labels file found. Complete labeling first.")
else:
    labels = pd.read_csv(labels_path)
    labels = labels.dropna(subset=["behavior"])
    print(f"Loaded {len(labels)} labeled frames")
    print(f"Classes: {labels['behavior'].unique()}")

    # Load corresponding features
    # (Simplified: assumes one feature file matched to the labels)
    feature_files = list(features_dir.rglob("*.parquet"))
    if feature_files and len(feature_files) > 0:
        features = pd.read_parquet(feature_files[0])
        
        # Match labels to features
        X = features.iloc[labels["frame"]].fillna(0)
        y = labels["behavior"]

        # Train/test split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
        )

        # Train XGBoost
        model = xgb.XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            random_state=RANDOM_STATE,
            eval_metric="mlogloss",
        )
        model.fit(X_train, y_train)

        # Evaluate
        y_pred = model.predict(X_test)
        print("\n=== Classification Report ===")
        print(classification_report(y_test, y_pred))
        print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")

        # Save model
        models_dir = Path(DRIVE_MODELS)
        models_dir.mkdir(parents=True, exist_ok=True)
        model_path = models_dir / "behavior_model.pkl"
        joblib.dump(model, model_path)
        print(f"\nModel saved to {model_path}")

In [ ]:
# Optional: Train with different window sizes
print("Run this cell to experiment with different window sizes.")
print("Currently uses single-frame features. Window-based features not yet implemented.")